# Get the data

In [151]:
import pandas as pd


df = pd.read_csv("../../../datasets/manufacturing.csv")
df

,Temperature (°C),Pressure (kPa),Temperature x Pressure,Material Fusion Metric,Material Transformation Metric,Quality Rating
0,209.762701,8.050855,1688.769167,44522.217074,9.229576e+06,99.999971
1,243.037873,15.812068,3842.931469,63020.764997,1.435537e+07,99.985703
2,220.552675,7.843130,1729.823314,49125.950249,1.072839e+07,99.999758
3,208.976637,23.786089,4970.736918,57128.881547,9.125702e+06,99.999975
4,184.730960,15.797812,2918.345014,38068.201283,6.303792e+06,100.000000
...,...,...,...,...,...,...
3952,156.811578,21.794290,3417.596965,34941.963896,3.855501e+06,100.000000
3953,197.850406,8.291704,1640.516924,39714.857236,7.744742e+06,99.999997
3954,241.357144,16.391910,3956.304672,62657.690952,1.405957e+07,99.989318
3955,209.040239,23.809936,4977.234763,57195.985528,9.134036e+06,99.999975


# Data Cleaning and EDA (Feature x Target)

#### Geral informations

In [152]:
print(f"The shape of dataset: {df.shape}, n_samples:{df.shape[0]}, n_features:{df.shape[1]}")
for name in df.columns:
    print(f"Column: {name} | dtype: {df[name].dtype} | {df[name].nunique()} differents values | Skew:{df[name].skew():.3f}")

The shape of dataset: (3957, 6), n_samples:3957, n_features:6
Column: Temperature (°C) | dtype: float64 | 3957 differents values | Skew:0.025
Column: Pressure (kPa) | dtype: float64 | 3957 differents values | Skew:0.010
Column: Temperature x Pressure | dtype: float64 | 3957 differents values | Skew:0.615
Column: Material Fusion Metric | dtype: float64 | 3957 differents values | Skew:0.344
Column: Material Transformation Metric | dtype: float64 | 3957 differents values | Skew:0.650
Column: Quality Rating | dtype: float64 | 3187 differents values | Skew:-4.476


#### Checking for invalid data in the dataset

In [153]:
print(f"Null values presents in the dataset: \n{df.isnull().sum()}\n")
print(f"Duplicated values presents in the dataset: {df.duplicated().sum()}")

Null values presents in the dataset: 
Temperature (°C)                  0
Pressure (kPa)                    0
Temperature x Pressure            0
Material Fusion Metric            0
Material Transformation Metric    0
Quality Rating                    0
dtype: int64

Duplicated values presents in the dataset: 0


#### Checking the features x targets relationships

In [155]:
df.describe()

,Temperature (°C),Pressure (kPa),Temperature x Pressure,Material Fusion Metric,Material Transformation Metric,Quality Rating
count,3957.000000,3957.000000,3957.000000,3957.000000,3.957000e+03,3957.000000
mean,200.034704,14.815558,2955.321308,48127.183128,1.003645e+07,96.260179
std,58.135717,5.772040,1458.224940,23812.213513,7.599356e+06,12.992262
min,100.014490,5.003008,513.706875,10156.971955,9.999462e+05,1.000000
25%,150.871296,9.692984,1798.247303,27626.929091,3.433810e+06,99.941129
50%,198.603371,14.832557,2678.277782,44611.452164,7.833390e+06,99.999997
75%,251.366552,19.749680,3929.058261,67805.443846,1.588251e+07,100.000000
max,299.992804,24.999132,7365.018714,103756.181544,2.699783e+07,100.000000


# Data Engineering

In [156]:
from sklearn.preprocessing import StandardScaler


for feature in features:
    scaler = StandardScaler()
    df[feature] = scaler.fit_transform(df[[feature]])



# Data tunning

#### Discretization

In [157]:
# How we see in the TARGET GRPAH SEARCH FOR OUTLIERS, THE MEDIUM/LOW are considering 'outliers', but if need to predict if some was with MEDIUN/LOW quality the model will wrong and we probabilite goes for house :(, so let's see better this


total = len(df)
top_value = df['Quality Rating'].max()
count_top = (df['Quality Rating'] == top_value).sum()

print(f"Total of samples: {total}")
print(f"Samples of high values: {count_top}")
print(f"High quality cases: {((df['Quality Rating'] > 80).sum()/total)*100:.2f}%")
print(f"Representability of high: {(count_top/total)*100:.2f}%")

Total of samples: 3957
Samples of high values: 540
High quality cases: 93.45%
Representability of high: 13.65%


In [158]:
bins = [-1, 70, 90, 100] 
labels = [0, 1, 2]
df['Quality Rating New'] = pd.cut(df['Quality Rating'], bins=bins, labels=labels)

print(f"Values before the discretization: {df['Quality Rating'].describe()}\n\n")
print(f"Values after the discretization: {df['Quality Rating New'].value_counts()}")


# LET'S UNDERSTAMDING WHAT HAPPEND:


"""
The data was 'squeezed' 93% above 80, meaning that for the model to have 93%+ accuracy, it only needed to guess a high value. 
However, for average quality (< 90) and poor quality (< 70), the model would be wrong, making it practically useless. 
Therefore, we divided the data so the model understands that there are poor quality values ​​and not just guess a high value:

0 -> poor quality
1 -> average quality
2 -> good quality
"""

Values before the discretization: count    3957.000000
mean       96.260179
std        12.992262
min         1.000000
25%        99.941129
50%        99.999997
75%       100.000000
max       100.000000
Name: Quality Rating, dtype: float64


Values after the discretization: Quality Rating New
2    3605
0     195
1     157
Name: count, dtype: int64


"\nThe data was 'squeezed' 93% above 80, meaning that for the model to have 93%+ accuracy, it only needed to guess a high value. \nHowever, for average quality (< 90) and poor quality (< 70), the model would be wrong, making it practically useless. \nTherefore, we divided the data so the model understands that there are poor quality values \u200b\u200band not just guess a high value:\n\n0 -> poor quality\n1 -> average quality\n2 -> good quality\n"

#### Best features

In [159]:
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

models = [ DecisionTreeClassifier(max_depth=10), RandomForestClassifier(max_depth=10, n_estimators = 150), XGBClassifier(), LGBMClassifier(verbose = -1)]

X = df.drop(columns= ['Quality Rating', 'Quality Rating New'], axis= 1)
y = df['Quality Rating New']
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size=0.2, random_state= 42, stratify=y)

In [160]:
# SFM METHODS


for model in models:
    print(f'=====================================')
    # See the features 
    sfm = SelectFromModel(model, threshold='mean')
    sfm.fit(X_train, y_train)
    mask = X.columns[sfm.get_support()]
    print(f"Model: {model.__class__.__name__}, features: {mask.tolist()}")
    print()

    # See the permutation

    model.fit(X_train, y_train)
    perm = permutation_importance(model, X, y, random_state= 42, n_repeats=10)
    for feature, importance in zip(X.columns, perm.importances_mean):
        print(f"Feature: {feature} | importance: {importance*100:.2f}")
    print()

    # See the report
    y_preds = model.predict(X_test)
    cr = classification_report(y_test, y_preds, output_dict=True)
    print(f"Classification report: {cr}")
    print(f'\n\n')

Model: DecisionTreeClassifier, features: ['Material Transformation Metric']

Feature: Temperature (°C) | importance: 4.87
Feature: Pressure (kPa) | importance: 0.00
Feature: Temperature x Pressure | importance: 0.00
Feature: Material Fusion Metric | importance: 0.00
Feature: Material Transformation Metric | importance: 16.16

Classification report: {'0': {'precision': 0.975, 'recall': 1.0, 'f1-score': 0.9873417721518988, 'support': 39.0}, '1': {'precision': 1.0, 'recall': 0.967741935483871, 'f1-score': 0.9836065573770492, 'support': 31.0}, '2': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 722.0}, 'accuracy': 0.9987373737373737, 'macro avg': {'precision': 0.9916666666666667, 'recall': 0.989247311827957, 'f1-score': 0.9903161098429827, 'support': 792.0}, 'weighted avg': {'precision': 0.9987689393939394, 'recall': 0.9987373737373737, 'f1-score': 0.9987350156472381, 'support': 792.0}}



Model: RandomForestClassifier, features: ['Temperature (°C)', 'Material Transformation

In [161]:
## LETS SEE IF WE DROP THE USELESS FEATURES ACCORDING THE LAST VIEW WE HAVE THE SAME ACCURACY

from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

models = [ DecisionTreeClassifier(max_depth=10), RandomForestClassifier(max_depth=10, n_estimators = 150), XGBClassifier(), LGBMClassifier(verbose = -1)]

X = df.drop(columns= ['Quality Rating', 'Quality Rating New', 'Pressure (kPa)', 'Temperature x Pressure', 'Material Fusion Metric'], axis= 1)
y = df['Quality Rating New']
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size=0.2, random_state= 42, stratify=y)

# SFM METHODS


for model in models:
    print(f'=====================================')
    # See the features 
    sfm = SelectFromModel(model, threshold='mean')
    sfm.fit(X_train, y_train)
    mask = X.columns[sfm.get_support()]
    print(f"Model: {model.__class__.__name__}, features: {mask.tolist()}")
    print()

    # See the permutation

    model.fit(X_train, y_train)
    perm = permutation_importance(model, X, y, random_state= 42, n_repeats=10)
    for feature, importance in zip(X.columns, perm.importances_mean):
        print(f"Feature: {feature} | importance: {importance*100:.2f}")
    print()

    # See the report
    y_preds = model.predict(X_test)
    cr1 = classification_report(y_test, y_preds, output_dict=True)
    print(f"Classification report: {cr}")
    print(f'\n\n')

Model: DecisionTreeClassifier, features: ['Temperature (°C)']

Feature: Temperature (°C) | importance: 16.16
Feature: Material Transformation Metric | importance: 4.87

Classification report: {'0': {'precision': 0.975, 'recall': 1.0, 'f1-score': 0.9873417721518988, 'support': 39.0}, '1': {'precision': 1.0, 'recall': 0.967741935483871, 'f1-score': 0.9836065573770492, 'support': 31.0}, '2': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 722.0}, 'accuracy': 0.9987373737373737, 'macro avg': {'precision': 0.9916666666666667, 'recall': 0.989247311827957, 'f1-score': 0.9903161098429827, 'support': 792.0}, 'weighted avg': {'precision': 0.9987689393939394, 'recall': 0.9987373737373737, 'f1-score': 0.9987350156472381, 'support': 792.0}}



Model: RandomForestClassifier, features: ['Temperature (°C)']

Feature: Temperature (°C) | importance: 11.67
Feature: Material Transformation Metric | importance: 4.91

Classification report: {'0': {'precision': 0.975, 'recall': 1.0, 'f1-score':

In [162]:
import pandas as pd

df1 = pd.DataFrame(cr).T
df2 = pd.DataFrame(cr1).T

print("Model with no select features")
print(df1)

print("\nModel with select features")
print(df2)

Model with no select features
              precision    recall  f1-score     support
0              0.975000  1.000000  0.987342   39.000000
1              1.000000  0.967742  0.983607   31.000000
2              1.000000  1.000000  1.000000  722.000000
accuracy       0.998737  0.998737  0.998737    0.998737
macro avg      0.991667  0.989247  0.990316  792.000000
weighted avg   0.998769  0.998737  0.998735  792.000000

Model with select features
              precision    recall  f1-score     support
0              0.975000  1.000000  0.987342   39.000000
1              0.967742  0.967742  0.967742   31.000000
2              1.000000  0.998615  0.999307  722.000000
accuracy       0.997475  0.997475  0.997475    0.997475
macro avg      0.980914  0.988786  0.984797  792.000000
weighted avg   0.997506  0.997475  0.997482  792.000000


So when we drop 3 features and metrics still the same, we do a GOOD WORK HERE :)

# Review about EDA and Preprocessing
- Descretization in the target in 3 (0, 1 and 2) (bad, medium and good)
- Select Features (Drop 'Pressure', 'Temperatura x Pressure', 'Material fusion Metric')
- Scaler in all the features
